# Hydrogen-Deuterium Exchange (HDX-MS) Prediction

<a href="https://colab.research.google.com/github/elkins/diff-hdx/blob/main/examples/hdx_prediction_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hydrogen-Deuterium Exchange Mass Spectrometry (HDX-MS) is a powerful technique for probing protein dynamics and solvent accessibility. Backbone amide protons continuously exchange with solvent protons; when a protein is placed in a D₂O buffer, these amide hydrogens are gradually replaced by deuterium.

The rate of exchange depends on:
1. **Intrinsic Exchange Rate ($k_{int}$)**: Depends on the local amino acid sequence, temperature, and pH (Bai et al., 1993).
2. **Protection Factor ($PF$)**: Represents how much the structural environment protects the amide proton from exchange.

The observable exchange rate $k_{obs}$ is given by:
$$ k_{obs} = rac{k_{int}}{PF} $$

In this tutorial, we will use the `diff_hdx` module to simulate these physical processes in a fully differentiable manner.

In [ ]:
# Setup for Google Colab
import sys

if "google.colab" in sys.modules:
    !pip install -q git+https://github.com/elkins/diff-hdx.git

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

# Set matplotlib style
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Intrinsic Exchange Rates ($k_{int}$)

The intrinsic exchange rate of an amide proton in a random-coil state is primarily governed by the side chains of the neighbouring residues (i-1 and i+1), as well as pH and temperature.

The `intrinsic_rates` function implements the empirical model described by **Bai et al. (1993)**.

In [ ]:
from diff_hdx.kernels import intrinsic_rates

# A synthetic sequence (20 residues)
sequence = "MKVTIYLADGSEPWCRYFNT"
residue_indices = jnp.arange(1, len(sequence) + 1)

# Compute intrinsic rates at standard conditions (pH 7.0, 20°C / 293.15 K)
k_int = intrinsic_rates(sequence, ph=7.0, temperature=293.15)

# Plot the intrinsic rates (log scale is typical for HDX rates)
plt.figure(figsize=(10, 4))
plt.bar(residue_indices, np.log10(np.array(k_int)), color="skyblue", edgecolor="black")
plt.xticks(residue_indices, list(sequence))
plt.xlabel("Residue")
plt.ylabel("log10(k_int) [min⁻¹]")
plt.title("Intrinsic Exchange Rates (Bai et al. 1993)")
plt.tight_layout()
plt.show()

## 2. Protection Factors ($PF$)

The actual exchange rate is slowed down by the protein's structure. Protons buried in the hydrophobic core or involved in stable hydrogen bonds exchange much slower. 

We model the Protection Factor using a phenomenological Linderstrøm-Lang model:
$$ \ln(PF) = eta_{asa} (1 - 	ext{SASA}) + eta_c N_{HB} $$

where SASA is the Solvent Accessible Surface Area, and $N_{HB}$ is the hydrogen bond energy/count.
Let's generate some synthetic 3D coordinates to represent our 20-residue sequence and compute the protection factors.

In [ ]:
from diff_hdx.kernels import h_bond_energy, protection_factors

# Generate a synthetic "folded" structure: a tight spiral
angles = jnp.linspace(0, 4 * jnp.pi, len(sequence))
z = jnp.linspace(-5.0, 5.0, len(sequence))
coords = jnp.stack([5.0 * jnp.cos(angles), 5.0 * jnp.sin(angles), z], axis=1)

# We'll use the same coordinates as both donors and acceptors for simplicity
h_bonds = h_bond_energy(coords, coords, cutoff=4.0)

# Calculate protection factors
# beta_asa and beta_c are scaling factors that are typically fitted to experimental data
pf = protection_factors(coords, h_bonds, beta_c=2.0, beta_asa=5.0)

plt.figure(figsize=(10, 4))
plt.plot(residue_indices, np.log10(np.array(pf)), marker="o", linestyle="-", color="purple")
plt.xticks(residue_indices, list(sequence))
plt.xlabel("Residue")
plt.ylabel("log10(Protection Factor)")
plt.title("Estimated Protection Factors (log scale)")
plt.tight_layout()
plt.show()

## 3. Deuterium Uptake Kinetics ($D(t)$)

Assuming the EX2 kinetic regime (structural opening/closing is much faster than the intrinsic chemical exchange), the fractional deuterium uptake at time $t$ is:
$$ D(t) = 1 - \exp\left(-rac{k_{int}}{PF} tight) $$

Let's simulate the uptake across several time points, typically measured in HDX-MS experiments (e.g., 10s, 1m, 10m, 1h).

In [ ]:
from diff_hdx.kernels import deuterium_uptake

# Typical HDX time points in minutes
time_points = [10 / 60, 1.0, 10.0, 60.0]  # 10s, 1m, 10m, 1h
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

plt.figure(figsize=(10, 5))

for t, color in zip(time_points, colors, strict=False):
    uptake = deuterium_uptake(pf, k_int, time=t)

    # "Butterfly plot" style:
    plt.plot(
        residue_indices,
        np.array(uptake),
        marker="o",
        linestyle="-",
        color=color,
        label=f"t = {t:.2f} min",
    )

plt.xticks(residue_indices, list(sequence))
plt.ylim(-0.05, 1.05)
plt.xlabel("Residue")
plt.ylabel("Fractional Deuterium Uptake $D(t)$")
plt.title("Simulated HDX-MS Uptake Profile")
plt.legend()
plt.tight_layout()
plt.show()

### Individual Uptake Curves
We can also look at the uptake kinetics for specific residues over a continuous time axis.

In [ ]:
t_cont = jnp.logspace(-2, 3, 100)  # 0.01 to 1000 mins

plt.figure(figsize=(8, 5))

# Plot a highly protected residue and a highly exposed residue
for res_idx in [5, 15]:
    # Reshape t_cont to broadcast over the single residue calculation
    # k_obs = k_int / pf
    k_obs_res = k_int[res_idx] / pf[res_idx]
    d_t = 1.0 - jnp.exp(-k_obs_res * t_cont)

    plt.plot(
        np.array(t_cont),
        np.array(d_t),
        linewidth=2,
        label=f"Residue {sequence[res_idx]} {res_idx + 1}",
    )

plt.xscale("log")
plt.ylim(-0.05, 1.05)
plt.xlabel("Time (min)")
plt.ylabel("Fractional Uptake")
plt.title("Kinetics of Individual Amide Protons")
plt.legend()
plt.tight_layout()
plt.show()